# Scale vs Configuration — Phase 2 technical canary

Reproducible Google Colab notebook for the frozen Phase 2 VLM canary. It executes **one model per Colab session**, first with one query and then by exact-prefix resumption through all 12 canary queries.

Before running:

1. In Colab, select **Runtime → Change runtime type → Runtime Version 2026.07 → GPU**.
2. Upload `phase2_stimuli_transport_v1.tar` to `MyDrive/scale-vs-configuration/phase2/artifacts/`.
3. Select exactly one `MODEL_KEY` in the next cell. Use a separate Colab session for each model.
4. Run cells in order. Do not reconnect or change the runtime between the one-query invocation and the resumption invocation.

This notebook contains no full-inference command. Canary outputs are technical compatibility evidence and are excluded from scientific analysis. If any technical step fails, stop and preserve the output files for review.

The Git ref `phase2-canary-v2` is intentionally required and must be published only after the notebook and validation artifacts are committed.

> Technical revision v2: the global Colab `pip check` is retained as diagnostic evidence but is not blocking. Exact versions, targeted imports, CUDA, model loading, frozen revisions, stimulus integrity, and runner validation remain blocking. The scientific design is unchanged.


In [ ]:
MODEL_KEY = "qwen3_vl_4b_instruct"  # @param ["qwen3_vl_4b_instruct", "internvl3_5_4b_hf", "llava_next_mistral_7b"]
CANARY_GIT_REF = "phase2-canary-v2"
REPOSITORY_URL = "https://github.com/grifo114/scale-vs-configuration.git"

ALLOWED_MODEL_KEYS = {
    "qwen3_vl_4b_instruct",
    "internvl3_5_4b_hf",
    "llava_next_mistral_7b",
}
assert MODEL_KEY in ALLOWED_MODEL_KEYS
print(f"Selected model: {MODEL_KEY}")
print(f"Required Git ref: {CANARY_GIT_REF}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tarfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

DRIVE_ROOT = Path("/content/drive/MyDrive/scale-vs-configuration/phase2")
DRIVE_ARCHIVE = DRIVE_ROOT / "artifacts/phase2_stimuli_transport_v1.tar"
LOCAL_ARCHIVE = Path("/content/phase2_stimuli_transport_v1.tar")
REPO_ROOT = Path("/content/scale-vs-configuration")
OUTPUT_ROOT = DRIVE_ROOT / "inference"
CACHE_DIR = DRIVE_ROOT / "hf_cache" / MODEL_KEY
PROVENANCE_DIR = DRIVE_ROOT / "provenance" / MODEL_KEY / "canary"

for directory in (DRIVE_ROOT / "artifacts", OUTPUT_ROOT, CACHE_DIR, PROVENANCE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, value):
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def run_checked(command, *, cwd=None, env=None, log_path=None):
    command = [str(item) for item in command]
    completed = subprocess.run(
        command, cwd=cwd, env=env, text=True, capture_output=True
    )
    print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="", file=sys.stderr)
    if log_path is not None:
        Path(log_path).write_text(
            "COMMAND\n" + " ".join(command) +
            "\n\nSTDOUT\n" + completed.stdout +
            "\nSTDERR\n" + completed.stderr,
            encoding="utf-8",
        )
    completed.check_returncode()
    return completed

print(f"Drive root: {DRIVE_ROOT}")
print(f"Persistent output root: {OUTPUT_ROOT}")
print(f"Persistent model cache: {CACHE_DIR}")

In [ ]:
import numpy as np
import torch

os_release = {}
for line in Path("/etc/os-release").read_text(encoding="utf-8").splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os_release[key] = value.strip().strip('\"')

base_runtime = {
    "captured_at_utc": utc_now(),
    "declared_colab_runtime_version": "2026.07",
    "python": platform.python_version(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "ubuntu": os_release.get("PRETTY_NAME"),
    "cuda_available": bool(torch.cuda.is_available()),
    "torch_cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "gpu_devices": [],
}

if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        base_runtime["gpu_devices"].append({
            "index": index,
            "name": props.name,
            "total_memory_bytes": props.total_memory,
            "compute_capability": [props.major, props.minor],
        })

write_json(PROVENANCE_DIR / "base_runtime.json", base_runtime)
print(json.dumps(base_runtime, indent=2))

assert base_runtime["python"] == "3.12.13", "Select Colab runtime 2026.07"
assert base_runtime["torch"].split("+", 1)[0] == "2.11.0"
assert base_runtime["numpy"] == "2.0.2"
assert base_runtime["ubuntu"] == "Ubuntu 22.04.5 LTS"
assert base_runtime["cuda_available"] is True
assert len(base_runtime["gpu_devices"]) >= 1
print("Base Colab runtime preflight: OK")

In [ ]:
if not REPO_ROOT.exists():
    run_checked([
        "git", "clone", "--branch", CANARY_GIT_REF, "--depth", "1",
        REPOSITORY_URL, str(REPO_ROOT),
    ])
elif not (REPO_ROOT / ".git").is_dir():
    raise RuntimeError(f"Existing path is not a Git worktree: {REPO_ROOT}")

remote_url = run_checked(
    ["git", "config", "--get", "remote.origin.url"], cwd=REPO_ROOT
).stdout.strip()
if remote_url not in {REPOSITORY_URL, REPOSITORY_URL.removesuffix(".git")} :
    raise RuntimeError(f"Unexpected repository remote: {remote_url}")

head = run_checked(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT).stdout.strip()
ref_commit = run_checked(
    ["git", "rev-parse", f"{CANARY_GIT_REF}^{{commit}}"], cwd=REPO_ROOT
).stdout.strip()
assert head == ref_commit
tracked_status = run_checked(
    ["git", "status", "--porcelain=v1", "--untracked-files=no"], cwd=REPO_ROOT
).stdout
assert tracked_status == ""

expected_hashes = {
    "configs/phase2_inference_protocol.json": "3c09ca97984f47b868758ce3b4d07bf85e38dabd3d66b7e4eae9bb474ce4b774",
    "configs/phase2_query_manifest.jsonl": "61e4f9a8583dc6f375b6d4d9f72361acf1d9ff3a976485bf5b2f83736ea0360e",
    "scripts/run_phase2_inference.py": "643a63625d97aa893322410cf865bd2aa190d2f9b5d4c984aa6c50ef09a38818",
    "scripts/package_phase2_stimuli.py": "aae1366fc05caf13a410676c7cea88df417cbeb9c82a3825288d887cba09b185",
    "configs/phase2_stimuli_transport_v1.json": "c3e88b6225d7026005c6419830fcb3f3ffda3d39ec64e0986c3e15cdac6f8fb0",
    "configs/phase2_colab_environment_v1.json": "4d2e96152a71a0105c2cdbf1ec817d1f3eaf75ed4ff98b1f05bb071deab9045c",
}
for relative_path, expected in expected_hashes.items():
    actual = sha256_file(REPO_ROOT / relative_path)
    print(f"{relative_path}: {actual}")
    assert actual == expected

environment_spec = json.loads(
    (REPO_ROOT / "configs/phase2_colab_environment_v1.json").read_text(encoding="utf-8")
)
protocol = json.loads(
    (REPO_ROOT / "configs/phase2_inference_protocol.json").read_text(encoding="utf-8")
)
assert environment_spec["status"] == "predeclared_pending_colab_preflight"
assert environment_spec["scientific_design_changed"] is False
assert environment_spec["canary_execution"]["full_mode_allowed"] is False
assert protocol["protocol_status"] == "predeclared_pending_canary"

git_identity = {
    "captured_at_utc": utc_now(),
    "requested_ref": CANARY_GIT_REF,
    "resolved_commit": head,
    "remote_url": remote_url,
}
write_json(PROVENANCE_DIR / "git_identity.json", git_identity)
print(json.dumps(git_identity, indent=2))
print("Frozen Git inputs: OK")

In [ ]:
pins = environment_spec["pip_install"]["packages_in_install_order"]
assert environment_spec["pip_install"]["upgrade_strategy"] == "only-if-needed"
assert environment_spec["pip_install"]["allow_prereleases"] is False

run_checked([
    sys.executable, "-m", "pip", "install",
    "--upgrade-strategy", "only-if-needed", *pins,
], log_path=PROVENANCE_DIR / "pip_install.log")

import importlib
import importlib.metadata
import subprocess

global_pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    check=False,
)

pip_check_parts = [
    part.rstrip()
    for part in (
        global_pip_check.stdout,
        global_pip_check.stderr,
    )
    if part.strip()
]
pip_check_text = "\n".join(pip_check_parts)
if pip_check_text:
    pip_check_text += "\n"

(PROVENANCE_DIR / "pip_check.log").write_text(
    pip_check_text,
    encoding="utf-8",
)

print(f"Global pip check return code: {global_pip_check.returncode}")
if pip_check_text:
    print(pip_check_text, end="")
if global_pip_check.returncode != 0:
    print(
        "Global pip check conflicts were recorded as diagnostic-only. "
        "The targeted inference stack remains subject to blocking checks."
    )

expected_versions = dict(item.rsplit("==", 1) for item in pins)

targeted_modules = {
    "huggingface-hub": "huggingface_hub",
    "tokenizers": "tokenizers",
    "safetensors": "safetensors",
    "sentencepiece": "sentencepiece",
    "pillow": "PIL",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "transformers": "transformers",
    "numpy": "numpy",
    "torch": "torch",
}

assert set(expected_versions).issubset(targeted_modules)

targeted_import_results = {}
for distribution, module_name in targeted_modules.items():
    try:
        importlib.import_module(module_name)
    except Exception as error:
        raise RuntimeError(
            f"Targeted import failed: {distribution} -> {module_name}"
        ) from error

    targeted_import_results[distribution] = {
        "module": module_name,
        "status": "ok",
    }
    print(f"Import OK: {distribution} -> {module_name}")

installed_versions = {
    package: importlib.metadata.version(package)
    for package in expected_versions
}

for package, expected in expected_versions.items():
    actual = installed_versions[package]
    print(f"{package}: {actual}")
    assert actual == expected, f"{package}: {actual} != {expected}"

assert importlib.metadata.version("numpy") == "2.0.2"
assert importlib.metadata.version("torch").split("+", 1)[0] == "2.11.0"

pip_check_report = {
    "schema_version": "1.0",
    "scope": "global_colab_environment_diagnostic_only",
    "blocking": False,
    "returncode": global_pip_check.returncode,
    "clean": global_pip_check.returncode == 0,
    "targeted_imports_completed": True,
    "targeted_imports": targeted_import_results,
    "pinned_package_versions": installed_versions,
}

write_json(
    PROVENANCE_DIR / "pip_check_report.json",
    pip_check_report,
)
write_json(
    PROVENANCE_DIR / "pinned_package_versions.json",
    installed_versions,
)

print("Targeted pinned package environment: OK")


In [ ]:
transport = environment_spec["stimulus_transport"]
assert DRIVE_ARCHIVE.is_file(), (
    f"Upload the archive before continuing: {DRIVE_ARCHIVE}"
)
assert DRIVE_ARCHIVE.stat().st_size == transport["archive_size_bytes"]
drive_archive_sha = sha256_file(DRIVE_ARCHIVE)
print(f"Drive archive SHA-256: {drive_archive_sha}")
assert drive_archive_sha == transport["archive_sha256"]

if not LOCAL_ARCHIVE.is_file() or sha256_file(LOCAL_ARCHIVE) != transport["archive_sha256"]:
    shutil.copyfile(DRIVE_ARCHIVE, LOCAL_ARCHIVE)

assert LOCAL_ARCHIVE.stat().st_size == transport["archive_size_bytes"]
local_archive_sha = sha256_file(LOCAL_ARCHIVE)
print(f"Local archive SHA-256: {local_archive_sha}")
assert local_archive_sha == transport["archive_sha256"]

manifest_rows = [
    json.loads(line)
    for line in (REPO_ROOT / "configs/phase2_query_manifest.jsonl")
        .read_text(encoding="utf-8").splitlines()
    if line.strip()
]
manifest_rows.sort(key=lambda row: int(row["run_index"]))
expected_member_names = [str(row["stimulus_path"]) for row in manifest_rows]
assert len(expected_member_names) == 588
assert len(set(expected_member_names)) == 588

existing_targets = [(REPO_ROOT / name).exists() for name in expected_member_names]
if any(existing_targets) and not all(existing_targets):
    raise RuntimeError("Partial stimulus extraction detected; restart with a fresh Colab VM")

with tarfile.open(LOCAL_ARCHIVE, mode="r:") as archive:
    members = archive.getmembers()
    actual_member_names = [member.name for member in members]
    assert len(members) == transport["entry_count"] == 588
    assert actual_member_names == expected_member_names
    for member in members:
        pure = PurePosixPath(member.name)
        assert member.isfile()
        assert not pure.is_absolute()
        assert ".." not in pure.parts
    if not any(existing_targets):
        archive.extractall(path=REPO_ROOT, members=members, filter="data")

assert all((REPO_ROOT / name).is_file() for name in expected_member_names)
transport_record = {
    "captured_at_utc": utc_now(),
    "drive_archive_path": str(DRIVE_ARCHIVE),
    "drive_archive_sha256": drive_archive_sha,
    "local_archive_path": str(LOCAL_ARCHIVE),
    "local_archive_sha256": local_archive_sha,
    "entry_count": len(expected_member_names),
}
write_json(PROVENANCE_DIR / "stimulus_transport_verification.json", transport_record)
print("Safe stimulus extraction: OK")

In [ ]:
runner_path = REPO_ROOT / "scripts/run_phase2_inference.py"
run_checked(
    [sys.executable, str(runner_path), "--project-root", str(REPO_ROOT), "--validate-only"],
    cwd=REPO_ROOT,
    log_path=PROVENANCE_DIR / "runner_validate_only.log",
)
print("Runner validation of all 588 stimuli: OK")

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
revision_checks = []
for model in protocol["models"]:
    info = api.model_info(model["model_id"], revision=model["revision"])
    check = {
        "model_key": model["key"],
        "model_id": model["model_id"],
        "expected_revision": model["revision"],
        "resolved_revision": info.sha,
        "matches": info.sha == model["revision"],
    }
    revision_checks.append(check)
    print(json.dumps(check, indent=2))
    assert check["matches"] is True

write_json(PROVENANCE_DIR / "frozen_revision_checks.json", {
    "captured_at_utc": utc_now(),
    "checks": revision_checks,
})
print("All three frozen model revisions resolve: OK")

In [ ]:
pip_freeze = run_checked([sys.executable, "-m", "pip", "freeze", "--all"]).stdout
(PROVENANCE_DIR / "pip_freeze.txt").write_text(pip_freeze, encoding="utf-8")

selected_model = next(model for model in protocol["models"] if model["key"] == MODEL_KEY)
preflight = {
    "schema_version": "1.0",
    "captured_at_utc": utc_now(),
    "model_key": MODEL_KEY,
    "model_id": selected_model["model_id"],
    "model_revision": selected_model["revision"],
    "git_ref": CANARY_GIT_REF,
    "git_commit": head,
    "base_runtime": base_runtime,
    "pinned_package_versions": installed_versions,
    "global_pip_check_returncode": global_pip_check.returncode,
    "global_pip_check_blocking": False,
    "targeted_imports_completed": True,
    "stimulus_archive_sha256": local_archive_sha,
    "runner_validate_only_completed": True,
    "all_frozen_revisions_resolved": all(item["matches"] for item in revision_checks),
    "full_mode_allowed": False,
}
write_json(PROVENANCE_DIR / "colab_preflight.json", preflight)
print(json.dumps(preflight, indent=2))
print("Persistent preflight capture: OK")

## First canary invocation: exactly one new query

Run this cell once. If it fails, do not retry, delete, or repair outputs. Preserve `metadata.json`, `failures.jsonl` when present, and the invocation log for review.

In [ ]:
model_output_dir = OUTPUT_ROOT / MODEL_KEY / "canary"
results_path = model_output_dir / "results.jsonl"
metadata_path = model_output_dir / "metadata.json"
failures_path = model_output_dir / "failures.jsonl"

existing_results = (
    [line for line in results_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if results_path.exists() else []
)
if failures_path.exists() and failures_path.stat().st_size > 0:
    raise RuntimeError(f"Existing technical failure requires review: {failures_path}")
if existing_results:
    raise RuntimeError(
        f"Expected no existing result before the first invocation; found {len(existing_results)}"
    )

runner_environment = os.environ.copy()
runner_environment["PYTHONHASHSEED"] = "0"
runner_environment["TOKENIZERS_PARALLELISM"] = "false"
runner_environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

runner_command = [
    sys.executable, str(runner_path),
    "--project-root", str(REPO_ROOT),
    "--model-key", MODEL_KEY,
    "--mode", "canary",
    "--output-root", str(OUTPUT_ROOT),
    "--cache-dir", str(CACHE_DIR),
]
run_checked(
    [*runner_command, "--max-new-queries", "1"],
    cwd=REPO_ROOT, env=runner_environment,
    log_path=PROVENANCE_DIR / "invocation_01_max_new_queries_1.log",
)

first_rows = [
    json.loads(line) for line in results_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
first_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
assert len(first_rows) == 1
assert first_metadata["status"] == "partial"
assert first_metadata["completed_queries"] == 1
assert not failures_path.exists() or failures_path.stat().st_size == 0
print("First invocation completed exactly 1/12 queries: OK")

## Same-session exact-prefix resumption

Without reconnecting or changing the runtime, run the next cell to resume from the stored one-row prefix and complete the remaining 11 canary queries.

In [ ]:
current_rows = [
    json.loads(line) for line in results_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
if failures_path.exists() and failures_path.stat().st_size > 0:
    raise RuntimeError(f"Technical failure requires review: {failures_path}")

if len(current_rows) == 12:
    print("Canary already contains 12 rows; no new inference was launched.")
elif len(current_rows) == 1:
    run_checked(
        runner_command, cwd=REPO_ROOT, env=runner_environment,
        log_path=PROVENANCE_DIR / "invocation_02_resume_to_12.log",
    )
else:
    raise RuntimeError(f"Expected an exact one-row prefix; found {len(current_rows)} rows")

completed_rows = [
    json.loads(line) for line in results_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
completed_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
assert len(completed_rows) == 12
assert completed_metadata["status"] == "complete"
assert completed_metadata["completed_queries"] == 12
assert not failures_path.exists() or failures_path.stat().st_size == 0
print("Exact-prefix resumption completed 12/12 canary queries: OK")

In [ ]:
canary_rows = sorted(
    [row for row in manifest_rows if row["is_canary"] is True],
    key=lambda row: int(row["canary_index"]),
)
assert len(canary_rows) == 12
assert len(completed_rows) == 12

allowed_response_statuses = {"valid", "invalid_json", "invalid_schema"}
runtime_fingerprints = set()
for sequence_index, (query, result) in enumerate(zip(canary_rows, completed_rows), start=1):
    assert result["sequence_index"] == sequence_index
    assert result["pair_id"] == query["pair_id"]
    assert result["run_index"] == query["run_index"]
    assert result["canary_index"] == query["canary_index"]
    assert result["scene_id"] == query["scene_id"]
    assert result["fold"] == query["fold"]
    assert result["distance_stratum"] == query["distance_stratum"]
    assert result["stimulus_path"] == query["stimulus_path"]
    assert result["stimulus_sha256"] == query["stimulus_sha256"]
    assert result["prompt"] == query["prompt"]
    assert result["prompt_sha256"] == hashlib.sha256(query["prompt"].encode("utf-8")).hexdigest()
    assert result["ground_truth_m"] == query["ground_truth_m"]
    assert result["model_key"] == MODEL_KEY
    assert result["model_id"] == selected_model["model_id"]
    assert result["model_revision"] == selected_model["revision"]
    assert result["protocol_sha256"] == expected_hashes["configs/phase2_inference_protocol.json"]
    assert result["query_manifest_sha256"] == expected_hashes["configs/phase2_query_manifest.jsonl"]
    assert result["runner_sha256"] == expected_hashes["scripts/run_phase2_inference.py"]
    assert isinstance(result["raw_response"], str)
    assert result["response_status"] in allowed_response_statuses
    runtime_fingerprints.add(result["runtime_fingerprint_sha256"])

assert len(runtime_fingerprints) == 1
assert runtime_fingerprints == {completed_metadata["runtime_fingerprint_sha256"]}
assert completed_metadata["mode"] == "canary"
assert completed_metadata["model_key"] == MODEL_KEY
assert completed_metadata["model_revision"] == selected_model["revision"]
assert completed_metadata["git"]["commit"] == head
assert completed_metadata["protocol_sha256"] == expected_hashes["configs/phase2_inference_protocol.json"]
assert completed_metadata["query_manifest_sha256"] == expected_hashes["configs/phase2_query_manifest.jsonl"]
assert completed_metadata["runner_sha256"] == expected_hashes["scripts/run_phase2_inference.py"]
assert completed_metadata["target_queries"] == 12
assert completed_metadata["valid_responses"] + completed_metadata["invalid_responses"] == 12

first_log_path = PROVENANCE_DIR / "invocation_01_max_new_queries_1.log"
resume_log_path = PROVENANCE_DIR / "invocation_02_resume_to_12.log"
assert first_log_path.is_file()
assert resume_log_path.is_file()
assert "Existing exact prefix: 0/12" in first_log_path.read_text(encoding="utf-8")
assert "Existing exact prefix: 1/12" in resume_log_path.read_text(encoding="utf-8")

validation_report = {
    "schema_version": "1.0",
    "validated_at_utc": utc_now(),
    "validation_scope": "technical_canary_only",
    "scientific_results_analyzed": False,
    "model_key": MODEL_KEY,
    "model_id": selected_model["model_id"],
    "model_revision": selected_model["revision"],
    "git_ref": CANARY_GIT_REF,
    "git_commit": head,
    "result_rows": len(completed_rows),
    "valid_json_responses": completed_metadata["valid_responses"],
    "invalid_responses_retained": completed_metadata["invalid_responses"],
    "runtime_fingerprint_sha256": completed_metadata["runtime_fingerprint_sha256"],
    "technical_failures_recorded": 0,
    "exact_prefix_resume_verified": True,
    "status": "complete_pending_cross_model_review",
}
write_json(PROVENANCE_DIR / "canary_validation.json", validation_report)

artifact_paths = [
    results_path, metadata_path,
    PROVENANCE_DIR / "base_runtime.json",
    PROVENANCE_DIR / "git_identity.json",
    PROVENANCE_DIR / "pip_install.log",
    PROVENANCE_DIR / "pip_check.log",
    PROVENANCE_DIR / "pip_check_report.json",
    PROVENANCE_DIR / "pip_freeze.txt",
    PROVENANCE_DIR / "pinned_package_versions.json",
    PROVENANCE_DIR / "stimulus_transport_verification.json",
    PROVENANCE_DIR / "runner_validate_only.log",
    PROVENANCE_DIR / "frozen_revision_checks.json",
    PROVENANCE_DIR / "colab_preflight.json",
    PROVENANCE_DIR / "invocation_01_max_new_queries_1.log",
    PROVENANCE_DIR / "invocation_02_resume_to_12.log",
    PROVENANCE_DIR / "canary_validation.json",
]
collection = {
    "schema_version": "1.0",
    "created_at_utc": utc_now(),
    "model_key": MODEL_KEY,
    "files": [
        {
            "path": str(path),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in artifact_paths
        if path.is_file()
    ],
    "failures_file": {
        "path": str(failures_path),
        "exists": failures_path.exists(),
        "size_bytes": failures_path.stat().st_size if failures_path.exists() else 0,
    },
}
write_json(PROVENANCE_DIR / "collection_manifest.json", collection)
print(json.dumps(validation_report, indent=2))
print(f"Collection manifest: {PROVENANCE_DIR / 'collection_manifest.json'}")
print("TECHNICAL CANARY COMPLETE. Do not run full inference before cross-model review and a tracked protocol transition.")

## Handoff

Preserve the complete model-specific inference and provenance directories in Google Drive. Repeat this notebook in a new Colab session for each remaining model. After all three technical canaries, collect the JSONL, metadata, failure records if any, logs, validation reports, and collection manifests for review.

Do not release the 588-query runs until the canary evidence has been examined and the protocol status has been updated in a traceable commit.